In [1]:
from torch.utils.data import DataLoader
from torch import nn
import torch


In [2]:
from torchvision.transforms import v2
from medmnist import PathMNIST

tf = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=(0.5,0.5,0.5),std=(0.5,0.5,0.5))
])


test_dataset = PathMNIST(root="./data/",split="test",transform=tf,download=True,size=64)

n_labels = len(test_dataset.info["label"].items())

In [3]:

num_workers = 4

test_batch_size = 128

test_dl = DataLoader(
    test_dataset,
    batch_size= test_batch_size,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True
)

In [4]:
from models.cross_predictor import Predictor

model_type = "cross_predictor"
model_type = "full_" + model_type

version = "2.4"

model = Predictor(n_labels=n_labels)
model.load_state_dict(torch.load(f"model_weights/finetuning/{model_type}/{version}/model_epoch_100.pt"))

print(model)


Predictor(
  (autoencoder): AutoEncoder(
    (patcher): CNN(
      (conv): Conv2d(3, 1024, kernel_size=(4, 4), stride=(4, 4))
      (fc): Linear(in_features=1024, out_features=512, bias=True)
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
    )
    (encoder): Encoder(
      (transformer_blocks): Sequential(
        (0): TransformerBlock(
          (mha): MultiHeadAttention(
            (linear_qkv): Linear(in_features=512, out_features=1536, bias=True)
            (linear_out): Linear(in_features=512, out_features=512, bias=True)
          )
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (ff): FeedForward(
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
            (gelu): GELU(approximate='none')
          )
          (droppath1

In [ ]:
loss = nn.CrossEntropyLoss(label_smoothing=0.0)

In [6]:
from training_functions import test
import mlflow

device = "cuda" if torch.cuda.is_available() else "cpu"

mlflow.set_tracking_uri("http://192.168.1.36:5000")
mlflow.set_experiment(model_type)
with mlflow.start_run(run_name=version):

        mlflow_dataset = mlflow.data.numpy_dataset.from_numpy(
                features=test_dataset.imgs,
                targets=test_dataset.labels
        )
        mlflow.log_input(mlflow_dataset, context="testing")

        mlflow.set_tags({
                "stage":"testing",
        })

        model = model.to(device)
        torch.set_float32_matmul_precision('high')
        model = torch.compile(model)
        test_loss,test_acc,test_auc = test(model, device, test_dl,None, loss, 0,"Testing")

        mlflow.log_metrics({
                "test_loss":test_loss,
                "test_acc":test_acc,
                "test_auc":test_auc
        })

Epoch 1: Testing:   0%|          | 0/57 [00:00<?, ?it/s]

🏃 View run 2.4 at: http://192.168.1.36:5000/#/experiments/5/runs/9a09173333e34061ac4d1310d6812465
🧪 View experiment at: http://192.168.1.36:5000/#/experiments/5
